<a href="https://colab.research.google.com/github/pedrodavidreyes/Python-Code/blob/main/Pipeline_completo_para_limpieza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline completo para limpieza de datos


1.   Combinar funciones pequeñas en un solo pipeline clean_data(df).
2.   Ordenar los pasos de limpieza: nombres → tipos → valores faltantes/invalidos.
3.   Escribir una función main() que carga → limpia → exporta el archivo final.

Un pipeline de limpieza es como una línea de producción:
Entra el dataset crudo y Pasa por una serie de etapas ordenadas:
1.   Estandarizar texto
2.   Convertir tipos
3.   Tratar valores inválidos y faltantes.
4.   Sale un dataset limpio, listo para el análisis.

Muy importante determinar que se hara con los sentinels antes de la limpieza, para no contaminar el diagnóstico.

In [ ]:

# En este caso se determina para numericos, convertirlos a nulos y para texto, convertirlos a "unknown"
# Importamos el dataset
import pandas as pd
df = pd.read_csv("/datasets/everpeak_retail.csv")
# Función para reemplazar sentinels en columnas numéricas y de texto
def reemplazar_sentinels_global(df, numeric_cols, text_cols):
    numeric_sentinels = [-999, 999] # Ejemplo de sentinels numéricos
    for col in numeric_cols: # Ciclo para columnas numéricas
        df[col] = pd.to_numeric(df[col], errors="coerce") # Convertir a numérico, forzando errores a NaN
        df[col] = df[col].replace(numeric_sentinels, pd.NA) # Reemplazar sentinels numéricos por valor faltante en pandas

    text_sentinels = ["?"] # Ejemplo de sentinels de texto
    for col in text_cols: # Ciclo para columnas de texto
        df[col] = df[col].replace(text_sentinels, "unknown") # Reemplazar sentinels de texto por "unknown"
    return df
# establecer columnas a procesar
columnas_numericas = ["customer_age"]      # columnas numéricas a procesar
columnas_texto = ["product_category"]      # columnas texto a procesar
# aplicar la función de reemplazo de sentinels al dataframe
df = reemplazar_sentinels_global(df, columnas_numericas, columnas_texto) # aplicar la función de reemplazo de sentinels al dataframe
# mostrar resultados
df.info()
print(df.head())

# Luego de reemplazar los sentinels, se puede proceder con el diagnóstico y la limpieza, sin que los sentinels contaminen el proceso.
# establecer columnas a procesar
cols_imputar_mediana = ["customer_age"]
cols_imputar_unknown = ["city", "state"]
cols_imputar_fecha = ["order_date"]
# crear función para imputar segun el diagnostico los ausentes
def imputar_segun_diagnostico(df, median_fill_cols, fill_unknown_cols, date_drop_cols):
    # rellenar con la mediana en columnas numéricas
    for col in median_fill_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce") # Convertir a numérico, forzando errores a NaN
        med = df[col].median() # Calcular la mediana de la columna
        df[col] = df[col].fillna(med) # Rellenar con la mediana en columnas numéricas

    # rellenar con texto "unknown" en columnas categóricas
    for col in fill_unknown_cols:
        df[col] = df[col].fillna("unknown") # Rellenar con "unknown" en columnas de texto

    # eliminar registros con valores ausentes en columnas tipo fecha
    for col in date_drop_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce") # Convertir a datetime, forzando errores a NaT
        df = df.dropna(subset=[col]).reset_index(drop=True) # eliminar filas con fechas ausentes y resetear índices
    return df

# aplicar función y mostrar resultados
df = imputar_segun_diagnostico(df, cols_imputar_mediana, cols_imputar_unknown, cols_imputar_fecha)
df.info()
print(df.head())
# guardar df_clean en un CSV nuevo
df_clean.to_csv("RUTA_AQUI/nombre_archivo.csv", index=False)


# NOTAS

*   Pipeline de limpieza: conjunto ordenado de pasos que transforman datos crudos en datos limpios.
*   Orden de operaciones de limpieza: secuencia en la que aplicas reglas (texto → tipos → valores inválidos).
*   clean_data(df) (nombre estándar): función central que ejecuta el pipeline.